# Interactive Maps — VIIRS Electrification Inequality

**Author:** Bouchra Daddaoui  
**Repository:** viirs-electrification-ml  

Interactive folium maps for exploring NTL radiance, LISA spatial clusters, and energy poverty classification  
across Brazil, China, and Morocco (including Western Sahara).

Each map is exported as a self-contained HTML file that can be opened in any browser.

**Outputs:**
- `figures/map_ntl_choropleth.html` — NTL radiance choropleth per country
- `figures/map_lisa_clusters.html` — LISA cluster types per country
- `figures/map_energy_poverty.html` — Energy poverty classification per country
- `figures/map_interactive_combined.html` — All countries & themes with layer toggle

In [ ]:
import sys
sys.path.insert(0, '../scripts')

import numpy as np
import geopandas as gpd
import folium
from shapely.geometry import box
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from spatial_analysis import build_knn_weights, local_moran
from interactive_maps import (
    ntl_map, lisa_map, energy_map, build_combined_map, save_map,
    ENERGY_COLOURS, LISA_COLOURS,
)

FIGURES = Path('../figures')
FIGURES.mkdir(exist_ok=True)

print('Libraries loaded.')

## 1. Synthetic Tile Data

Spatially correlated synthetic tiles matching real VIIRS panel structure.  
Morocco bbox extends to 20.8°N to include Western Sahara.

In [ ]:
def generate_country_tiles(country, n_tiles, bbox, ntl_mean, ntl_std, seed):
    rng = np.random.default_rng(seed)
    minx, miny, maxx, maxy = bbox
    cols = int(np.sqrt(n_tiles))
    rows = n_tiles // cols

    xs = np.linspace(minx, maxx, cols + 1)
    ys = np.linspace(miny, maxy, rows + 1)

    geoms, tile_ids = [], []
    for i in range(rows):
        for j in range(cols):
            geoms.append(box(xs[j], ys[i], xs[j+1], ys[i+1]))
            tile_ids.append(f"{country[:3].upper()}_{i:02d}_{j:02d}")

    n = len(geoms)
    centroids = np.array([[g.centroid.x, g.centroid.y] for g in geoms])
    dists = np.linalg.norm(centroids[:, None] - centroids[None, :], axis=-1)
    cov = ntl_std**2 * np.exp(-dists / (0.3 * (maxx - minx)))
    ntl = rng.multivariate_normal(np.full(n, ntl_mean), cov)
    ntl = np.clip(ntl, 0, None)

    pop_density = rng.lognormal(mean=np.log(100), sigma=1.2, size=n)

    gdf = gpd.GeoDataFrame({
        'tile_id': tile_ids,
        'country': country,
        'ntl_mean': ntl,
        'pop_density': pop_density,
    }, geometry=geoms, crs='EPSG:4326')
    return gdf


countries_cfg = [
    dict(country='Brazil',  n_tiles=100, bbox=(-48, -23, -43, -18), ntl_mean=12.5, ntl_std=8.0,  seed=1),
    dict(country='China',   n_tiles=100, bbox=(116,  29, 122,  33), ntl_mean=28.3, ntl_std=14.0, seed=2),
    dict(country='Morocco', n_tiles=100, bbox=(-17.1, 20.8, -1.0, 35.9), ntl_mean=8.1, ntl_std=5.5, seed=3),
]

gdfs = {cfg['country']: generate_country_tiles(**cfg) for cfg in countries_cfg}

for c, g in gdfs.items():
    print(f"{c:8s}: {len(g)} tiles | NTL mean={g.ntl_mean.mean():.2f} | std={g.ntl_mean.std():.2f}")

## 2. Compute LISA Clusters and Energy Classes

LISA clusters from Local Moran's I (p < 0.05).  
Energy poverty thresholds applied per-country at the 25th and 75th NTL percentiles.

In [ ]:
def add_lisa_labels(gdf):
    w = build_knn_weights(gdf, k=8)
    lisa_df = local_moran(gdf['ntl_mean'].values, w, significance=0.05)
    gdf = gdf.copy()
    gdf['cluster_label'] = lisa_df['cluster_label'].values
    return gdf


def add_energy_class(gdf):
    ntl = gdf['ntl_mean']
    q25 = ntl.quantile(0.25)
    q75 = ntl.quantile(0.75)
    gdf = gdf.copy()
    gdf['energy_class'] = np.where(
        ntl >= q75, 'Electrified',
        np.where(ntl >= q25, 'Energy poor', 'Low pop / unlit')
    )
    return gdf


gdfs = {c: add_energy_class(add_lisa_labels(g)) for c, g in gdfs.items()}

for c, g in gdfs.items():
    print(f"\n{c}:")
    print(g['cluster_label'].value_counts().to_string())
    print(g['energy_class'].value_counts().to_string())

## 3. NTL Choropleth Maps (Plasma Colormap)

One interactive map per country — NTL radiance intensity coloured with the plasma palette.  
Hover a tile to see its ID, country, and NTL value.

In [ ]:
for country, gdf in gdfs.items():
    m = ntl_map(gdf, country)
    save_map(m, FIGURES / f'map_ntl_{country.lower()}.html')

# Combined NTL across all countries on one map
import json
from interactive_maps import choropleth_layer

m_ntl = folium.Map(location=[20, 10], zoom_start=2, tiles='CartoDB positron')
for country, gdf in gdfs.items():
    choropleth_layer(gdf, 'ntl_mean', f'NTL — {country}', cmap_name='plasma').add_to(m_ntl)
folium.LayerControl(collapsed=False).add_to(m_ntl)
save_map(m_ntl, FIGURES / 'map_ntl_choropleth.html')

## 4. LISA Cluster Maps

Spatial cluster types from Local Moran's I — HH (hotspot), LL (coldspot), HL/LH (outliers).

In [ ]:
from interactive_maps import lisa_layer

m_lisa = folium.Map(location=[20, 10], zoom_start=2, tiles='CartoDB positron')
for country, gdf in gdfs.items():
    lisa_layer(gdf, f'LISA — {country}').add_to(m_lisa)

folium.LayerControl(collapsed=False).add_to(m_lisa)

# Legend
from interactive_maps import _add_legend_html
_add_legend_html(m_lisa, LISA_COLOURS, 'LISA Clusters')

save_map(m_lisa, FIGURES / 'map_lisa_clusters.html')

## 5. Energy Poverty Classification Maps

Three-class scheme based on NTL percentile thresholds: Electrified / Energy poor / Low pop or unlit.

In [ ]:
from interactive_maps import energy_poverty_layer

m_ep = folium.Map(location=[20, 10], zoom_start=2, tiles='CartoDB positron')
for country, gdf in gdfs.items():
    energy_poverty_layer(gdf, f'Energy Poverty — {country}').add_to(m_ep)

folium.LayerControl(collapsed=False).add_to(m_ep)
_add_legend_html(m_ep, ENERGY_COLOURS, 'Energy Classification')

save_map(m_ep, FIGURES / 'map_energy_poverty.html')

## 6. Combined Multi-Country Interactive Map

All three countries, all three themes in a single map with a layer toggle panel.  
Toggle layers on/off to compare NTL intensity, spatial clusters, and energy poverty side by side.

In [ ]:
m_combined = build_combined_map(gdfs)
save_map(m_combined, FIGURES / 'map_interactive_combined.html')

print()
print('Interactive maps saved. Open figures/map_interactive_combined.html in a browser.')

## Summary

| File | Description |
|---|---|
| `figures/map_ntl_choropleth.html` | NTL radiance intensity, plasma palette, all 3 countries |
| `figures/map_lisa_clusters.html` | LISA spatial cluster types (HH/LL/HL/LH) |
| `figures/map_energy_poverty.html` | Energy poverty classification (3 classes) |
| `figures/map_interactive_combined.html` | All themes + countries, layer toggle panel |